In [1]:
from datetime import date
from collections import Counter, defaultdict
import warnings
import os

import periodictable
import h5py
import numpy as np

import qcportal
from qcportal.external import scaffold
from qcportal.molecules import Molecule
from qcportal.singlepoint import SinglepointDriver, QCSpecification
from qcportal.optimization import OptimizationSpecification
from qcelemental.models.procedures import OptimizationProtocols
from qcelemental.physical_constants import constants

ADDRESS = "https://api.qcarchive.molssi.org:443"
qc_client = qcportal.PortalClient(
    ADDRESS, 
    cache_dir=".",
)

/Users/jenniferclark/mamba/envs/qca/lib/python3.11/site-packages/nglview/__init__.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
dataset = qc_client.get_dataset("optimization", "OpenFF Organometallic Complexes Architector Minimum Energy Structures v0.0")

In [ ]:
spec = QCSpecification(
    program='psi4',
    driver=SinglepointDriver.gradient,
    method='BP86',
    basis='def2-TZVP',
    keywords={
        'maxiter': 500, 
        'scf_properties': ['dipole', 'quadrupole', 'wiberg_lowdin_indices', 'mayer_indices', 'lowdin_charges', 'mulliken_charges'],
        'function_kwargs': {'properties': ['dipole_polarizabilities']},
    },
    protocols={'wavefunction': 'none'}
)
opt_spec = OptimizationSpecification(
    program="geometric",
    qc_specification=spec, 
    keywords={
        "tmax": 0.3,
        "check": 0,
        "qccnv": False,
        "reset": True,
        "trust": 0.1,
        "molcnv": False,
        "enforce": 0.0,
        "epsilon": 1e-05,
        "maxiter": 300,
        "coordsys": "dlc",
        "converge": ['energy',
                     '1e-4',],
    },
)
dataset.add_specification(name="BP86/def2-TZVP high energy threshold", specification=opt_spec)

InsertMetadata(error_description=None, errors=[], inserted_idx=[0], existing_idx=[])

In [3]:
scaffold.to_json(dataset, "scaffold_add_highE.json", compress=True)
#dataset.submit()

## Make Outputs

In [ ]:
for spec, obj in dataset.specifications.items():
    obj = obj.dict()
    obj = obj['specification']
    print(obj.keys())
    print("* Spec:", spec)
    print(f"    * program: {obj['program']}")
    print(f"    * keywords:")
    for k, v in obj["keywords"].items():
        print(f"       * {k}: {v}")
    print(f"    * qc_specification:")
    for k, field in obj['qc_specification'].items():
        print(f"       * {k}: {field}")
    print("* SCF properties:")
    for field in obj['qc_specification']['keywords']["scf_properties"]:
        print(f"       * {field}")

dict_keys(['program', 'qc_specification', 'keywords', 'protocols'])
* Spec: BP86/def2-TZVP
    * program: geometric
    * keywords:
       * tmax: 0.3
       * check: 0
       * qccnv: False
       * reset: True
       * trust: 0.1
       * molcnv: False
       * enforce: 0.0
       * epsilon: 1e-05
       * maxiter: 300
       * coordsys: dlc
       * convergence_set: GAU
    * qc_specification:
       * program: psi4
       * driver: SinglepointDriver.deferred
       * method: bp86
       * basis: def2-tzvp
       * keywords: {'maxiter': 500, 'scf_properties': ['dipole', 'quadrupole', 'wiberg_lowdin_indices', 'mayer_indices', 'lowdin_charges', 'mulliken_charges'], 'function_kwargs': {'properties': ['dipole_polarizabilities']}}
       * protocols: {}
* SCF properties:
       * dipole
       * quadrupole
       * wiberg_lowdin_indices
       * mayer_indices
       * lowdin_charges
       * mulliken_charges
dict_keys(['program', 'qc_specification', 'keywords', 'protocols'])
* Spec: BP86